<a href="https://colab.research.google.com/github/cmmm976/LyricsExplorer/blob/text-generation/lyrics_generator/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://towardsdatascience.com/fine-tuning-gpt2-for-text-generation-using-pytorch-2ee61a4f1ba7

In [1]:
!pip install git+https://github.com/huggingface/transformers
!pip install sqlalchemy
!pip install mysql-connector-python
!pip install datasets

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-drgphhyx
  Running command git clone -q https://github.com/huggingface/transformers /tmp/pip-req-build-drgphhyx
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
    Preparing wheel metadata ... done
     |████████████████████████████████| 596 kB 14.8 MB/s 
     |████████████████████████████████| 3.3 MB 63.8 MB/s 
     |████████████████████████████████| 59 kB 5.9 MB/s 
     |████████████████████████████████| 895 kB 45.5 MB/s 
  Created wheel for transformers: filename=transformers-4.13.0.dev0-py3-none-any.whl size=3101536 sha256=508d5af7ff0730102d04dd0f4bd35a579f9b09aa9130bfcc1aed90428f75ffe9
  Stored in directory: /tmp/pip-ephem-wheel-cache-0e_xpn5y/wheels/35/2e/a7/d819e3310040329f0f47e57c9e3e7a7338aa5e74c49acfe522
Successfully built transformers
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 3.13
    Uninstalling PyYAML-3.13:
      Successfully uni

In [2]:
!git clone https://github.com/huggingface/transformers.git

Cloning into 'transformers'...
remote: Enumerating objects: 88468, done.
remote: Counting objects: 100% (321/321), done.
remote: Compressing objects: 100% (205/205), done.
remote: Total 88468 (delta 158), reused 179 (delta 86), pack-reused 88147
Receiving objects: 100% (88468/88468), 71.52 MiB | 27.66 MiB/s, done.
Resolving deltas: 100% (63649/63649), done.


In [4]:
import os
os.environ['HOST'] = 'lyricsexplorer-db-prod.ckcggcvsoqgq.eu-west-3.rds.amazonaws.com'
os.environ['USER'] = 'admin_le_dev'
os.environ['PASSWORD'] = '3uTp6PHqT3qDG52'
os.environ['PORT'] = '3306'
os.environ['DATABASE'] = 'songs'

In [5]:
from sqlalchemy import *
import pandas as pd

engine = create_engine('mysql+mysqlconnector://'+os.getenv('USER')+':'+os.getenv('PASSWORD')+"@"
    +os.getenv('HOST')+':'+str(os.getenv('PORT'))+'/'+os.getenv('DATABASE'))
connection = engine.connect()

songs = pd.read_sql('songs',connection)

In [6]:
songs = songs[songs['artist'] != 'Blue Virus']
songs = songs[songs['lyrics'].notna()]
songs.reset_index(inplace=True,drop=True)

In [7]:
def clean_text(text):

    #remove... something
    text = text.replace('urlcopyembedcopy','')
    text = text.replace('embedshare','')

    #remove numbers

    text = re.sub(r'[0-999999]', '', text)

    return text

def get_artist_lyrics(artist,df):
  s = ''
  for i in range(len(df['lyrics'])):
      if df.loc[i,'artist'] == artist:
        s += "\n<BOS>\n"+ df.loc[i,'cleaned_lyrics'] + "\n<EOS>"
  return s

def get_all_lyrics(df):
  s = ''
  for i in range(len(df['lyrics'])):
        s += "\n<BOS>\n"+ df.loc[i,'cleaned_lyrics'] + "\n<EOS>"
  return s

In [8]:
import re

songs['cleaned_lyrics'] = [clean_text(item) for item in songs['lyrics']]

In [ ]:
vald_songs = songs[songs['artist'] == 'Vald']
vald_songs.reset_index(inplace = True, drop = True)

In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(vald_songs,test_size = 0.1, random_state = 42)

In [ ]:
train_set.reset_index(inplace=True,drop=True)
test_set.reset_index(inplace=True,drop=True)

In [ ]:
training_file = open("/content/training_file.txt", "w")
test_file = open("/content/test_file.txt", "w")

training_file.write(get_artist_lyrics('Vald',train_set))
test_file.write(get_artist_lyrics('Vald',test_set))

training_file.close()
test_file.close()

In [14]:
import torch
torch.cuda.empty_cache()

In [ ]:
!python /content/transformers/examples/pytorch/language-modeling/run_clm.py \
    --model_name_or_path asi/gpt-fr-cased-small \
    --train_file /content/training_file.txt \
    --validation_file /content/test_file.txt \
    --do_train \
    --do_eval \
		--num_train_epochs=5 \
		--per_device_train_batch_size=1 \
		--per_device_eval_batch_size=1 \
    --output_dir /content/models/vald \
		--overwrite_output_dir

10/02/2021 14:51:00 - WARNING - __main__ - Process rank: -1, device: cuda:0, n_gpu: 1distributed training: False, 16-bits training: False
10/02/2021 14:51:00 - INFO - __main__ - Training/evaluation parameters TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_pin_memory=True,
ddp_find_unused_parameters=None,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=True,
eval_accumulation_steps=None,
eval_steps=None,
evaluation_strategy=IntervalStrategy.NO,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
gradient_accumulation_steps=1,
gradient_checkpointing=False,
greater_is_better=None,
group_by_length=False,
hub_model_id=None,
hub_strategy=HubStrategy.EVERY_SAVE,
hub_token=<HUB_TOKEN>,
ignore_data_skip=False,
label_names=None,
label_smoothing_factor=0.0,
learning_rate=5e-05,
length_column_name=length,
load_best_model_at

In [ ]:
!python /content/transformers/examples/pytorch/text-generation/run_generation.py \
  --model_type gpt2 \
  --model_name_or_path /content/models/vald \
  --length 500 \
  --prompt "<BOS>" \
  --stop_token "<EOS>" \
  --k 150 \
  --p 0.95 \
  --num_return_sequences 5

10/02/2021 15:14:32 - WARNING - __main__ - device: cuda, n_gpu: 1, 16-bits training: False
10/02/2021 15:14:36 - INFO - __main__ - Namespace(device=device(type='cuda'), fp16=False, k=150, length=500, model_name_or_path='/content/models/vald', model_type='gpt2', n_gpu=1, no_cuda=False, num_return_sequences=5, p=0.95, padding_text='', prefix='', prompt='<BOS>', repetition_penalty=1.0, seed=42, stop_token='<EOS>', temperature=1.0, xlm_language='')
=== GENERATED SEQUENCE 1 ===
<BOS>, pussy pussy! pussy! pussy! pussy! pussy! pussy! pussy! pussy! pussy! oui, ouais, ouais, oui, ouais, ouais, ouais, ouais, ouais, ouais, ouais, ouais, ouais, oh oui, oh ouais, oh ouais, ah ouais, oh ouais, oh, oh, ah ouais, ouais, ouais, oh, non, oh, oh, oh, ouais, ah ouais, ouais, ouais, oh, ouais, ouais, oh, ah ouais, ouais, oh, oh, oh, ouais, oh, ouais, ouais, oh, ouais, ah ouais, ouais, oh, ouais, ouais, ouais, oh, ouais, oh, oui, ouais, ah oui, oh, ouais, ouais, oh, ouais, oh, ouais, oh, ouais, oh, ouais, a

In [9]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(songs,test_size = 0.1, random_state = 42)

In [10]:
train_set.reset_index(inplace=True,drop=True)
test_set.reset_index(inplace=True,drop=True)

In [11]:
training_file = open("/content/training_file_all_artists.txt", "w")
test_file = open("/content/test_file_all_artists.txt", "w")

training_file.write(get_all_lyrics(train_set))
test_file.write(get_all_lyrics(test_set))

training_file.close()
test_file.close()

In [17]:
!python /content/transformers/examples/pytorch/language-modeling/run_clm.py \
    --model_name_or_path asi/gpt-fr-cased-small \
    --train_file /content/training_file_all_artists.txt \
    --validation_file /content/test_file_all_artists.txt \
    --do_train \
    --do_eval \
		--num_train_epochs=5 \
		--per_device_train_batch_size=3 \
		--per_device_eval_batch_size=3 \
    --output_dir /content/models/all \
		--overwrite_output_dir

11/04/2021 12:59:38 - WARNING - __main__ - Process rank: -1, device: cuda:0, n_gpu: 1distributed training: False, 16-bits training: False
11/04/2021 12:59:38 - INFO - __main__ - Training/evaluation parameters TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_pin_memory=True,
ddp_find_unused_parameters=None,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=True,
eval_accumulation_steps=None,
eval_steps=None,
evaluation_strategy=IntervalStrategy.NO,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
gradient_accumulation_steps=1,
gradient_checkpointing=False,
greater_is_better=None,
group_by_length=False,
hub_model_id=None,
hub_strategy=HubStrategy.EVERY_SAVE,
hub_token=<HUB_TOKEN>,
ignore_data_skip=False,
label_names=None,
label_smoothing_factor=0.0,
learning_rate=5e-05,
length_column_name=length,
load_best_model_at

In [20]:
!python /content/transformers/examples/pytorch/text-generation/run_generation.py \
  --model_type gpt2 \
  --model_name_or_path /content/models/all \
  --length 500 \
  --prompt "<BOS> J'veux pas faire partie de la jet set" \
  --stop_token "<EOS>" \
  --k 150 \
  --p 0.95 \
  --num_return_sequences 5

11/04/2021 15:09:34 - WARNING - __main__ - device: cuda, n_gpu: 1, 16-bits training: False
11/04/2021 15:09:40 - INFO - __main__ - Namespace(device=device(type='cuda'), fp16=False, k=150, length=500, model_name_or_path='/content/models/all', model_type='gpt2', n_gpu=1, no_cuda=False, num_return_sequences=5, p=0.95, padding_text='', prefix='', prompt="<BOS> J'veux pas faire partie de la jet set", repetition_penalty=1.0, seed=42, stop_token='<EOS>', temperature=1.0, xlm_language='')
=== GENERATED SEQUENCE 1 ===
<BOS> J'veux pas faire partie de la jet set mais les gens m'rappelle
J'trouve ça juste étrange
J'sais même pas comment je vais faire pour rester sur la touche
J'viens d'Marseille tout ça c'est ma première fois, une fois qu'j'sors
J'vois que des négros s'entretuer toute la nuit, même après minuit
T'es dans la norme, mon son est de Mala dans l'boule
J'connais pas ta famille, ils ont mis leurs vies à prix, ils ont même pas payé tes droits
Nan c'est d'retour dans dix piges, mon son n'